# Fundamentals of Machine Learning - Exercise 9
Goal of this excercise is to complete the hands-on experience of the classification task.

## Household Prices Dataset
https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data

* ... I bet that you already know the data pretty well 😅

![meme03](https://github.com/rasvob/VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/blob/master/images/fml_09_meme_03.jpg?raw=true)

**Important attributes description:**
* SalePrice: The property's sale price in dollars. This is the target variable that you're trying to predict.
* MSSubClass: The building class
* BldgType: Type of dwelling
* HouseStyle: Style of dwelling
* OverallQual: Overall material and finish quality
* OverallCond: Overall condition rating
* YearBuilt: Original construction date
* Heating: Type of heating
* CentralAir: Central air conditioning
* GrLivArea: Above grade (ground) living area square feet
* BedroomAbvGr: Number of bedrooms above basement level)



In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, auc
from sklearn.preprocessing import OrdinalEncoder

# 🎯 Our goal is to predict if the house will be sold for more than 250k USD or not
* We will use categorized price as a **Target** variable

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/rasvob/VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/master/datasets/zsu_cv1_data.csv', sep=',')
df = df.loc[:, ['SalePrice','MSSubClass','BldgType','HouseStyle','OverallQual','OverallCond','YearBuilt','Heating','CentralAir','GrLivArea','BedroomAbvGr']]
df.loc[:, ['Target']] = (df.SalePrice > 250000).astype(int)
df = df.drop(['SalePrice'], axis=1)

In [3]:
df.head()

,MSSubClass,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,Heating,CentralAir,GrLivArea,BedroomAbvGr,Target
0,60,1Fam,2Story,7,5,2003,GasA,Y,1710,3,0
1,20,1Fam,1Story,6,8,1976,GasA,Y,1262,3,0
2,60,1Fam,2Story,7,5,2001,GasA,Y,1786,3,0
3,70,1Fam,2Story,7,5,1915,GasA,Y,1717,3,0
4,60,1Fam,2Story,8,5,2000,GasA,Y,2198,4,0


# Take a look at the features
* We will need it to answer the questions

In [4]:
df.describe()

,MSSubClass,OverallQual,OverallCond,YearBuilt,GrLivArea,BedroomAbvGr,Target
count,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,56.897260,6.099315,5.575342,1971.267808,1515.463699,2.866438,0.148630
std,42.300571,1.382997,1.112799,30.202904,525.480383,0.815778,0.355845
min,20.000000,1.000000,1.000000,1872.000000,334.000000,0.000000,0.000000
25%,20.000000,5.000000,5.000000,1954.000000,1129.500000,2.000000,0.000000
50%,50.000000,6.000000,5.000000,1973.000000,1464.000000,3.000000,0.000000
75%,70.000000,7.000000,6.000000,2000.000000,1776.750000,3.000000,0.000000
max,190.000000,10.000000,9.000000,2010.000000,5642.000000,8.000000,1.000000


## Categorial features EDA

In [5]:
df.describe(exclude=np.number)

,BldgType,HouseStyle,Heating,CentralAir
count,1460,1460,1460,1460
unique,5,8,6,2
top,1Fam,1Story,GasA,Y
freq,1220,726,1428,1365


### BldgType

In [16]:
df.BldgType.value_counts()
df = pd.concat([df, pd.get_dummies(df['BldgType'], prefix='BldgType')], axis=1).drop('BldgType', axis=1)
df

,MSSubClass,HouseStyle,OverallQual,OverallCond,YearBuilt,Heating,CentralAir,GrLivArea,BedroomAbvGr,Target,BldgType_1Fam,BldgType_2fmCon,BldgType_Duplex,BldgType_Twnhs,BldgType_TwnhsE
0,60,2Story,7,5,2003,GasA,Y,1710,3,0,True,False,False,False,False
1,20,1Story,6,8,1976,GasA,Y,1262,3,0,True,False,False,False,False
2,60,2Story,7,5,2001,GasA,Y,1786,3,0,True,False,False,False,False
3,70,2Story,7,5,1915,GasA,Y,1717,3,0,True,False,False,False,False
4,60,2Story,8,5,2000,GasA,Y,2198,4,0,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,2Story,6,5,1999,GasA,Y,1647,3,0,True,False,False,False,False
1456,20,1Story,6,6,1978,GasA,Y,2073,3,0,True,False,False,False,False
1457,70,2Story,7,9,1941,GasA,Y,2340,4,1,True,False,False,False,False
1458,20,1Story,5,6,1950,GasA,Y,1078,2,0,True,False,False,False,False


### HouseStyle

In [17]:
df.HouseStyle.value_counts()
df = pd.concat([df, pd.get_dummies(df['HouseStyle'], prefix='HouseStyle')], axis=1).drop('HouseStyle', axis=1)
df

,MSSubClass,OverallQual,OverallCond,YearBuilt,Heating,CentralAir,GrLivArea,BedroomAbvGr,Target,BldgType_1Fam,...,BldgType_Twnhs,BldgType_TwnhsE,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl
0,60,7,5,2003,GasA,Y,1710,3,0,True,...,False,False,False,False,False,False,False,True,False,False
1,20,6,8,1976,GasA,Y,1262,3,0,True,...,False,False,False,False,True,False,False,False,False,False
2,60,7,5,2001,GasA,Y,1786,3,0,True,...,False,False,False,False,False,False,False,True,False,False
3,70,7,5,1915,GasA,Y,1717,3,0,True,...,False,False,False,False,False,False,False,True,False,False
4,60,8,5,2000,GasA,Y,2198,4,0,True,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,6,5,1999,GasA,Y,1647,3,0,True,...,False,False,False,False,False,False,False,True,False,False
1456,20,6,6,1978,GasA,Y,2073,3,0,True,...,False,False,False,False,True,False,False,False,False,False
1457,70,7,9,1941,GasA,Y,2340,4,1,True,...,False,False,False,False,False,False,False,True,False,False
1458,20,5,6,1950,GasA,Y,1078,2,0,True,...,False,False,False,False,True,False,False,False,False,False


### Heating

In [19]:
df.Heating.value_counts()
df = pd.concat([df, pd.get_dummies(df['Heating'], prefix='Heating')], axis=1).drop('Heating', axis=1)
df

,MSSubClass,OverallQual,OverallCond,YearBuilt,CentralAir,GrLivArea,BedroomAbvGr,Target,BldgType_1Fam,BldgType_2fmCon,...,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl,Heating_Floor,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall
0,60,7,5,2003,Y,1710,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
1,20,6,8,1976,Y,1262,3,0,True,False,...,False,False,False,False,False,True,False,False,False,False
2,60,7,5,2001,Y,1786,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
3,70,7,5,1915,Y,1717,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
4,60,8,5,2000,Y,2198,4,0,True,False,...,False,True,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,6,5,1999,Y,1647,3,0,True,False,...,False,True,False,False,False,True,False,False,False,False
1456,20,6,6,1978,Y,2073,3,0,True,False,...,False,False,False,False,False,True,False,False,False,False
1457,70,7,9,1941,Y,2340,4,1,True,False,...,False,True,False,False,False,True,False,False,False,False
1458,20,5,6,1950,Y,1078,2,0,True,False,...,False,False,False,False,False,True,False,False,False,False


## Missing values

In [9]:
df.isna().sum()

,0
MSSubClass,0
BldgType,0
HouseStyle,0
OverallQual,0
OverallCond,0
YearBuilt,0
Heating,0
CentralAir,0
GrLivArea,0
BedroomAbvGr,0


## Labels distribution

In [14]:
df.Target.value_counts()

,count
Target,
0,1243
1,217


# ✅ Task (2p)
Complete the following tasks:

1. 📈 Describe what operations you are performing for each of the features
    * Mainly focus on categorical features
      
2. 📌 Answer the following questions:
    * **How many values are missing?** None
    * **How many instances do you have in each of the classes?**
      217 are Target, 1243 are Target
    * 🔎 **Which metric score do you propose for the classification model performance evaluation?** Data skewed to favor non targets => f1 score,
      accuracy would be missleading
          
3. ⚡Finish your preprocessing pipeline and split the data into the Input and Output part (i.e. `X` and `y` variables)

4. 🌳 Start with the **Decision Tree**
    * Use 5-fold cross validation
    * 🔎 Will you use *standard* cross validation or *stratified* cross validation?
    * Compute mean of the obtained score values
      
5. 🚀 Select one other algorithm from https://scikit-learn.org/stable/supervised_learning.html
    * Repeat the 5-fold CV
      
6. 📒 **Write down which default model is better**

7. 📊 Experiment with hyper-parameters
    * Select at least one important parameter for the model
    * Set the parameter value range
        * You can use random values, interval of values, ...
    * Do the 5-fold CV
        * Compute mean of the obtained score values
    * Document the experiment results using tables and/or plots
    * Describe the results in a Markdown cell

8. 📒 **Write down which model (default or tuned) is the best and why**

* **Document everything you do in a Markdown cells**
    * ❌ Results interpretation figured in real-time during task check is not allowed! ❌

##One-Hot Encoding
* new binary col for each unique value
* when you are using algo that works based on ordinality (higher num, higher rank) or distance, and the data cannot be interpreted like that -> for example ports of embarkation

##Label Encoding
* each label gets a number
* like decks labeled A,B,C etc. have nums assigned and transformed into decks 0,1,2


In [15]:
#preprocessing
#one-hot encoding everything is my solution to encoding reqs
df = pd.concat([df, pd.get_dummies(df['BldgType'], prefix='BldgType')], axis=1).drop('BldgType', axis=1)
df = pd.concat([df, pd.get_dummies(df['HouseStyle'], prefix='HouseStyle')], axis=1).drop('HouseStyle', axis=1)
df = pd.concat([df, pd.get_dummies(df['Heating'], prefix='Heating')], axis=1).drop('Heating', axis=1)



,MSSubClass,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,Heating,CentralAir,GrLivArea,BedroomAbvGr,Target
0,60,1Fam,2Story,7,5,2003,GasA,Y,1710,3,0
1,20,1Fam,1Story,6,8,1976,GasA,Y,1262,3,0
2,60,1Fam,2Story,7,5,2001,GasA,Y,1786,3,0
3,70,1Fam,2Story,7,5,1915,GasA,Y,1717,3,0
4,60,1Fam,2Story,8,5,2000,GasA,Y,2198,4,0
...,...,...,...,...,...,...,...,...,...,...,...
1455,60,1Fam,2Story,6,5,1999,GasA,Y,1647,3,0
1456,20,1Fam,1Story,6,6,1978,GasA,Y,2073,3,0
1457,70,1Fam,2Story,7,9,1941,GasA,Y,2340,4,1
1458,20,1Fam,1Story,5,6,1950,GasA,Y,1078,2,0


In [23]:
#except CentralAir, that one is binary Y/N, so ordinal is used
df.CentralAir.value_counts()
ca_cats = ['N', 'Y']
enc_ca = OrdinalEncoder(categories=[ca_cats])
df.loc[:, 'CentralAir'] = enc_ca.fit_transform(df[['CentralAir']])


In [37]:
mask_rows = (df['CentralAir'] == 0)
df.loc[mask_rows, 'CentralAir'] #rows, cols

,CentralAir
29,0.0
30,0.0
39,0.0
52,0.0
61,0.0
...,...
1387,0.0
1393,0.0
1412,0.0
1443,0.0


In [38]:
#data spliting into results - X and data - y
X, y = df.loc[:, df.columns != 'Target'], df.loc[:, 'Target'] #X training data, y lbl data


In [39]:
#test and train data split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=13)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1168, 26), (292, 26), (1168,), (292,))

In [40]:
#Tree training
#stratifiedKFold will balance the amount of classes in each fold
#ensuring even distribution of target variable
#so a situtation where one or more folds simply does not have one target class
#(and thus skewing results) does not happen
skf = StratifiedKFold(n_splits=5)
scores = list()
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    clf = DecisionTreeClassifier(random_state=13)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    scores.append(f1_score(y_test, y_pred))
    print(f'Target ratio in train set: {y_train.value_counts(normalize=True)[1]:.2}; Target ratio in test set: {y_test.value_counts(normalize=True)[1]:.2}')

scores

Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15
Target ratio in train set: 0.15; Target ratio in test set: 0.15


[0.6666666666666666,
 0.7764705882352941,
 0.5679012345679012,
 0.7333333333333333,
 0.6987951807228916]

##Random forest
*

In [ ]:
#
skf = StratifiedKFold(n_splits=5)
scores = list()
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    clf = RandomForestClassifier(random_state=13)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    scores.append(f1_score(y_test, y_pred))
    print(f'Target ratio in train set: {y_train.value_counts(normalize=True)[1]:.2}; Target ratio in test set: {y_test.value_counts(normalize=True)[1]:.2}')

scores